In [44]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict

In [56]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
FANTOM_INITIAL = DATA_INITIAL / "fantom_cat"
PREPROCESSING = Path("../preprocessing")

HIMORNA_INDEX_V2 = PREPROCESSING / "himorna_index-v2"
HIMORNA_GENES_SIGN =  PREPROCESSING / "himorna_genes_sign"
HIMORNA_GENES_SIGN_V2 =  PREPROCESSING / "himorna_genes_sign-v2"

IPS_INDEX_V2 = PREPROCESSING / "deg_index-v2/ips_deg_index"
FIBRO_INDEX_V2 = PREPROCESSING / "deg_index-v2/fibro_deg_index"

MATRICES_V2 = PREPROCESSING / "matrices-v2"
IPS_MATRICES = PREPROCESSING / "matrices/ips_matrices"
IPS_MATRICES_V2 = MATRICES_V2 / "ips_matrices"
FIBRO_MATRICES_V2 = MATRICES_V2 / "fibro_matrices"

INFO = PREPROCESSING / "info"
INFO_V2 = PREPROCESSING / "info-v2"
IPS_INFO = INFO / "ips_info"
FIBRO_INFO = INFO / "fibro_info"

IPS_INFO_V2 = INFO_V2 / "ips_info"
FIBRO_INFO_V2 = INFO_V2 / "fibro_info"

In [46]:
to_file = FANTOM / 'fantom_l2-genes_with_name-hg38-all_info.tsv'
fantom_l2_genes_update_with_names_all_info = pd.read_csv(to_file, sep="\t", index_col=False)
fantom_l2_genes_update_with_names_all_info.head(3)

,chrom,start,end,ens_gene,score,strand,new-ens_gene,new-gene_name
0,chr1,159537,162485,CATG00000042730,.,-,CATG00000042730,CATG00000042730
1,chr1,273882,274416,CATG00000042731,.,-,CATG00000042731,CATG00000042731
2,chr1,588120,634563,CATG00000042732,.,-,CATG00000042732,CATG00000042732


In [90]:
mask = fantom_l2_genes_update_with_names_all_info["new-ens_gene"] == "ENSG00000154783"
fantom_l2_genes_update_with_names_all_info[mask]

,chrom,start,end,ens_gene,score,strand,new-ens_gene,new-gene_name
73960,chr3,14810543,14938309,ENSG00000154783,.,+,ENSG00000154783,FGD5


In [47]:
len(fantom_l2_genes_update_with_names_all_info)

123997

In [48]:
fantom_l2_genes_update_with_names_only_new = fantom_l2_genes_update_with_names_all_info[["new-ens_gene","new-gene_name"]].drop_duplicates().reset_index(drop=True)
fantom_l2_genes_update_with_names_only_new.head(3)

,new-ens_gene,new-gene_name
0,CATG00000042730,CATG00000042730
1,CATG00000042731,CATG00000042731
2,CATG00000042732,CATG00000042732


In [49]:
len(fantom_l2_genes_update_with_names_only_new)

121449

In [50]:
mask = fantom_l2_genes_update_with_names_only_new["new-ens_gene"].apply(lambda x: "retired" not in x)
fantom_l2_genes_update_with_names_only_new = fantom_l2_genes_update_with_names_only_new[mask].reset_index(drop=True)
len(fantom_l2_genes_update_with_names_only_new)

121447

In [51]:
fantom_lv2_hg38 = fantom_l2_genes_update_with_names_only_new[["new-ens_gene"]]

In [52]:
fantom_lv2_hg38

,new-ens_gene
0,CATG00000042730
1,CATG00000042731
2,CATG00000042732
3,ENSG00000248527
4,ENSG00000225972
...,...
121442,CATG00000115125
121443,CATG00000114976
121444,CATG00000115126
121445,CATG00000114977


In [60]:
to_himorna = HIMORNA_GENES_SIGN / "H3K9me3/H3K9me3-ENSG00000260871-minus-genes_list-.tsv.gz"
corr_gene_ids_v1 = np.asarray(pd.read_csv(to_himorna, sep="\t", usecols=["genes_list"])["genes_list"].unique())
len(corr_gene_ids_v1)

13297

In [61]:
to_himorna = HIMORNA_GENES_SIGN_V2 / "H3K9me3/H3K9me3-ENSG00000260871-minus-genes_list-.tsv.gz"
corr_gene_ids_v2 = np.asarray(pd.read_csv(to_himorna, sep="\t", usecols=["genes_list"])["genes_list"].unique())
len(corr_gene_ids_v2)

13105

In [64]:
len(set(corr_gene_ids_v1) - set(corr_gene_ids_v2))

232

In [65]:
len(set(corr_gene_ids_v2) - set(corr_gene_ids_v1))

40

In [69]:
def create_matrix(to_deg, to_himorna, exp):
    """
    Воспринимать стоит так: есть реальная разметка классов - результаты нокдауна. Есть предсказания "алгорим himorna".
    Нужно оценить эффективность алгоритма himorna для данной lncRNA и метки.
    """
    hm = to_himorna.parent.stem
    ens_gene = to_himorna.stem.split("-")[1]
    
    corr_gene_ids = np.asarray(pd.read_csv(to_himorna, sep="\t", usecols=["genes_list"])["genes_list"].unique())
    deg_gene_ids = np.asarray(pd.read_csv(to_deg, sep="\t", usecols=["DEG.new-ens_gene"])["DEG.new-ens_gene"].unique())
    
    deg_mask = fantom_lv2_hg38["new-ens_gene"].isin(deg_gene_ids)
    corr_mask = fantom_lv2_hg38["new-ens_gene"].isin(corr_gene_ids)
    TP =  deg_mask &  corr_mask
    FP = ~deg_mask &  corr_mask
    FN =  deg_mask & ~corr_mask
    TN = ~deg_mask & ~corr_mask

    #Jaccard index
    #jc_index = (deg_mask & corr_mask).sum() / (deg_mask | corr_mask).sum()
    
    to_save = PREPROCESSING / f"matrices-v2/{exp}_matrices" / to_himorna.name.split('-')[0] / f"{to_himorna.name.split('-')[1]}-{to_deg.name.split('-')[1]}-{to_himorna.name.split('-')[2]}-{to_deg.name.split('-')[2]}-.npy"
    if to_save.exists():
        print(f"Skipped: ({to_himorna.name.split('-')[0]}) {to_himorna.name.split('-')[1]} {to_deg.name.split('-')[1]} {to_himorna.name.split('-')[2]} {to_deg.name.split('-')[2]}")
        return None
    to_save.parent.mkdir(exist_ok=True, parents=True)
    np.save(to_save, np.stack((TP, FP, FN, TN)))

In [70]:
with Parallel(n_jobs=-1) as pool:
        chunks = pool(
            delayed(create_matrix)(to_deg, to_himorna, exp)
            for exp in ["fibro", "ips"]
            for to_deg in (PREPROCESSING / f"deg_index-v2/{exp}_deg_index").glob(('*/*.tsv.gz'))
            for corr_sign in ("plus", "minus")
            for to_himorna in HIMORNA_GENES_SIGN_V2.glob(f'*/*-{to_deg.name.split("-")[0]}-{corr_sign}*-.tsv.gz')
        )

In [71]:
def number_of_plus_minus_genes(hm, lnc):
    subdf_list = []
    for to_subdf in HIMORNA_GENES_SIGN_V2.glob(f"{hm}/{hm}-{lnc}*"):
        subdf = pd.read_csv(to_subdf)
        subdf["sign"] = [to_subdf.stem.split("-")[2] for i in range(len(subdf))]
        subdf_list.append(subdf)
    subdf = pd.concat(subdf_list)
    plus = len(subdf[subdf["sign"] == "plus"])
    minus = len(subdf[subdf["sign"] == "minus"])
    return (plus, minus)

In [72]:
def number_of_up_down_genes(lnc, aso, exp):
    subdf_list = []
    for to_subdf in PREPROCESSING.joinpath(f"deg_index-v2/{exp}_deg_index").glob(f"{lnc}/{lnc}-{aso}*"):
        subdf = pd.read_csv(to_subdf, sep="\t")
        subdf_list.append(subdf)
    subdf = pd.concat(subdf_list)
    up = len(subdf[subdf["up_or_down_expr"] == "up"])
    down = len(subdf[subdf["up_or_down_expr"] == "down"])
    return (up, down)

In [73]:
number_of_plus_minus_genes("H3K27ac", "ENSG00000223813")

(2809, 668)

In [74]:
number_of_up_down_genes(lnc="ENSG00000223813", aso="G0223813_02", exp="ips")

(100, 41)

In [75]:
np.load(IPS_MATRICES_V2 / "H3K27ac" / "ENSG00000223813-G0223813_02-plus-up-.npy").sum(axis=1)

array([     2,   2807,     98, 118540])

In [76]:
np.load(IPS_MATRICES_V2 / "H3K27ac" / "ENSG00000223813-G0223813_02-plus-up-.npy")[0]

array([False, False, False, ..., False, False, False])

In [77]:
list_of_hm = ['H3K27ac', 'H3K27me3', 'H3K36me3', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K79me2', 'H3K9ac', 'H3K9me3', 'H4K20me1', 'methylation']

In [78]:
ENSG00000229847_conf_matrixes = [i for i in (FIBRO_MATRICES_V2 / "H3K9me3").glob("ENSG00000229847*.npy")]
ENSG00000229847_conf_matrixes

[PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_03-plus-up-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_03-minus-up-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_01-plus-down-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_01-minus-down-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_01-plus-up-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_01-minus-up-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_03-plus-down-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_03-minus-down-.npy'),
 PosixPath('../preprocessing/matrices-v2/fibro_matrices/H3K9me3/ENSG00000229847-G0229847_AD_02-plus-up-.npy'),
 PosixPath('.

In [79]:
def job(hm, exp):
    agg_info_by_hm = defaultdict(list)
    for to_matrix in PREPROCESSING.joinpath(f"matrices-v2/{exp}_matrices").glob(f"{hm}/*"):
        hm, lnc, aso, corr, expr =  to_matrix.parent.stem, to_matrix.stem.split("-")[0], to_matrix.stem.split("-")[1], to_matrix.stem.split("-")[2], to_matrix.stem.split("-")[3]
        matrix = np.load(to_matrix)
        TP = fantom_lv2_hg38[matrix[0]].shape[0]#deg_mask & corr_mask
        TN = fantom_lv2_hg38[matrix[1]].shape[0]#deg_mask & ~corr_mask
        FP = fantom_lv2_hg38[matrix[2]].shape[0]#~deg_mask & corr_mask
        FN = fantom_lv2_hg38[matrix[3]].shape[0]#~(deg_mask | corr_mask)
        
        up, down = number_of_up_down_genes(lnc=lnc, aso=aso, exp=exp)
        plus, minus = number_of_plus_minus_genes(hm=hm, lnc=lnc)
    
        expr_dict = {"up":up, "down":down}
        corr_dict = {"plus":plus, "minus":minus}
        
        agg_info_by_hm["hm"].append(hm)
        agg_info_by_hm["lnc"].append(lnc)
        agg_info_by_hm["aso"].append(aso)
        agg_info_by_hm["expr"].append(expr)
        agg_info_by_hm['corr'].append(corr)
        agg_info_by_hm["number_of_DEG"].append(expr_dict[expr])
        agg_info_by_hm["number_genes_from_himorna"].append(corr_dict[corr])
    
        agg_info_by_hm["TP"].append(TP)
        agg_info_by_hm["TN"].append(TN)
        agg_info_by_hm["FP"].append(FP)
        agg_info_by_hm["FN"].append(FN)
    df = pd.DataFrame(agg_info_by_hm).sort_values(by="aso")
    to_save = PREPROCESSING / f"info-v2/{exp}_info"  / f"{hm}.tsv"
    if to_save.exists():
        print(f"Skipped: {exp} {hm}")
        return
    to_save.parent.mkdir(exist_ok=True, parents=True)
    df.to_csv(to_save, sep="\t", index=False)

In [80]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(job)(hm=hm, exp=exp)
        for exp in ["fibro", "ips"]
        for hm in list_of_hm)

In [81]:
#проверка
to_csv= FIBRO_INFO_V2  / "H4K20me1.tsv"
pd.read_csv(to_csv, sep="\t").query("aso == 'G0163364_06'")

,hm,lnc,aso,expr,corr,number_of_DEG,number_genes_from_himorna,TP,TN,FP,FN
0,H4K20me1,ENSG00000163364,G0163364_06,down,minus,19,32,0,32,19,121396
1,H4K20me1,ENSG00000163364,G0163364_06,down,plus,19,24,0,24,19,121404


In [82]:
#проверка
to_csv= FIBRO_INFO_V2  / "H3K9me3.tsv"
pd.read_csv(to_csv, sep="\t").query("lnc == 'ENSG00000229847'").to_csv("H3K9me3-ENSG00000229847-all_ASO-conf_matrixes.tsv", sep="\t", index=False)

In [83]:
pd.concat([pd.read_csv(to_deg, sep="\t") for to_deg in FIBRO_INDEX_V2.glob((f'*/*ENSG00000163364-G0163364_06-down*'))]).shape

(19, 10)

In [84]:
pd.concat([pd.read_csv(to_himorna, sep="\t") for to_himorna in HIMORNA_GENES_SIGN_V2.glob((f'H4K20me1/*-ENSG00000163364-plus*'))]).shape

(24, 1)

In [85]:
pd.concat([pd.read_csv(to_himorna, sep="\t") for to_himorna in HIMORNA_GENES_SIGN_V2.glob((f'H4K20me1/*-ENSG00000163364-minus*'))]).shape

(32, 1)

In [86]:
matrix = [np.load(to_matrix) for to_matrix in FIBRO_MATRICES_V2.glob("H4K20me1/*ENSG00000163364-G0163364_06-minus-down*")][0]
print(fantom_lv2_hg38[matrix[0]].shape[0], fantom_lv2_hg38[matrix[1]].shape[0],
      fantom_lv2_hg38[matrix[2]].shape[0], fantom_lv2_hg38[matrix[3]].shape[0])

0 32 19 121396


In [87]:
matrix = [np.load(to_matrix) for to_matrix in FIBRO_MATRICES_V2.glob("H4K20me1/*ENSG00000163364-G0163364_06-plus-down*")][0]
print(fantom_lv2_hg38[matrix[0]].shape[0], fantom_lv2_hg38[matrix[1]].shape[0],
      fantom_lv2_hg38[matrix[2]].shape[0], fantom_lv2_hg38[matrix[3]].shape[0])

0 24 19 121404
